# The Jacobian Lens: Reading the Model's Global Workspace

**Goal**: After this session, you will be able to compute an averaged Jacobian lens for a transformer layer, read it, steer with it, patch lens coordinates, and measure how much of an activation lives in the verbalizable "J-space".
**Time**: 60 minutes.

## What and Why

Anthropic's workspace paper (Transformer Circuits, 2026) asks which internal representations a model can *verbally report*. Their tool is the **Jacobian lens**: instead of unembedding an intermediate activation directly (logit lens), first map it through the expected Jacobian to the final layer — "how would this activation move the final residual stream?" The lens's rows define concept directions that support steering, patching, and a sparse "J-space" that behaves like a global-workspace bottleneck (~25 active concepts at a time).

## Key Formula

$$J_\ell = \mathbb{E}_{t' \ge t}\left[\frac{\partial h_{\mathrm{final},\,t'}}{\partial h_{\ell,\,t}}\right] \qquad\qquad \mathrm{lens}(h) = \mathrm{softmax}\big(W_U\,\mathrm{norm}(J_\ell\, h)\big)$$

**Where:**
- $h_{\ell,t}$ — residual stream at layer $\ell$, position $t$ (shape `[d_model]`)
- $h_{\mathrm{final},t'}$ — final-layer residual stream at a downstream position $t' \ge t$
- $J_\ell$ — a `[d_model, d_model]` matrix: the Jacobian averaged over causally-valid position pairs and a prompt corpus
- $W_U$ — unembedding matrix `[vocab, d_model]`; row $u_t$ is token $t$'s readout vector
- $\mathrm{norm}$ — the final layer norm (parameter-free here, applied inside the lens read)

## Component Breakdown

- **Full Jacobian** of the model tail via autograd, then a causal-pair average → $J_\ell$
- **Lens read**: one function that is the logit lens at $M = I$ and the J-lens at $M = J_\ell$
- **J-lens vectors** $v_t = J_\ell^\top u_t$: reading direction = steering direction (chain rule)
- **Lens-coordinate patching**: swap two concepts' coordinates via the pseudoinverse
- **J-space decomposition**: greedy non-negative pursuit → what fraction of $h$ is verbalizable

## References

[Verbalizable Representations Form a Global Workspace in Language Models](https://transformer-circuits.pub/2026/workspace/index.html) (Transformer Circuits, 2026)

In [ ]:
# Setup: tiny causal transformer + probe utilities (all plumbing provided -- start at Part 1)
import torch
import torch.nn as nn
import torch.nn.functional as F
from jaxtyping import Float
from torch import Tensor

torch.manual_seed(0)
torch.set_default_dtype(torch.float64)  # we verify first-order identities numerically; fp32 eps would drown them

D_MODEL, N_HEADS, N_LAYERS, VOCAB, SEQ = 32, 4, 4, 100, 6
N_PROMPTS = 8   # the "corpus" the averaged Jacobian is estimated over
LAYER = 2       # the probed layer: h_layer = residual stream after block 2


class Block(nn.Module):
    """Pre-LN transformer block with causal attention (operates on a single [seq, d] sequence)."""

    def __init__(self, d: int, n_heads: int):
        super().__init__()
        self.n_heads = n_heads
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.proj = nn.Linear(d, d, bias=False)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x: Float[Tensor, "seq d"]) -> Float[Tensor, "seq d"]:
        s, d = x.shape
        q, k, v = self.qkv(self.ln1(x)).chunk(3, dim=-1)
        q, k, v = (t.view(s, self.n_heads, -1).transpose(0, 1) for t in (q, k, v))
        attn = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.proj(attn.transpose(0, 1).reshape(s, d))
        return x + self.mlp(self.ln2(x))


class TinyGPT(nn.Module):
    """4-block causal LM. Same tensor roles as a real decoder LM, just small enough for exact Jacobians.

    The tail ends at the last block's residual stream (h_final); the final norm is folded
    into the lens read, matching the paper's lens(h) = softmax(W_U norm(J h)).
    """

    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB, D_MODEL)
        self.pos_emb = nn.Embedding(SEQ, D_MODEL)
        self.blocks = nn.ModuleList(Block(D_MODEL, N_HEADS) for _ in range(N_LAYERS))
        self.W_U = nn.Linear(D_MODEL, VOCAB, bias=False)  # unembedding; .weight is [vocab, d_model]

    def hiddens(self, tokens: Float[Tensor, "seq"]) -> list:
        """Residual stream at every depth: hiddens[i] = state entering block i; hiddens[N_LAYERS] = h_final."""
        h = self.tok_emb(tokens) + self.pos_emb(torch.arange(tokens.shape[0]))
        states = [h]
        for blk in self.blocks:
            h = blk(h)
            states.append(h)
        return states

    def forward_from(self, layer: int, h: Float[Tensor, "seq d"]) -> Float[Tensor, "seq d"]:
        """Run the model tail: apply blocks[layer:] to h. forward_from(LAYER, hiddens[LAYER]) == h_final."""
        for blk in self.blocks[layer:]:
            h = blk(h)
        return h


model = TinyGPT()
for p in model.parameters():
    p.requires_grad_(False)  # we differentiate w.r.t. activations, not weights

prompts = torch.randint(0, VOCAB, (N_PROMPTS, SEQ))  # same role as a batch of tokenized corpus prompts
W_U_w = model.W_U.weight                             # [VOCAB, D_MODEL]


def h_at(tokens: Float[Tensor, "seq"], layer: int = LAYER) -> Float[Tensor, "seq d"]:
    """Residual stream at `layer` (detached: a fresh leaf you can differentiate the tail against)."""
    return model.hiddens(tokens)[layer].detach()


# NOTE: the model is random-init, so lens reads are not semantically meaningful --
# but every mathematical property of the method is exact, and those are what you practice here.
print("setup ok:", tuple(h_at(prompts[0]).shape))

## Part 1: The Averaged Jacobian $J_\ell$
Compute the full position-wise Jacobian of the model tail, then average the causally-valid blocks — this single matrix is the lens everything else reads through.

**Predict before you code**: The full Jacobian has SEQ × SEQ = 36 blocks of shape `[d, d]`, indexed by (t_out, t_in). What fraction of those blocks is *exactly* zero, and why? Commit the fraction as the first line of the stub.

<details><summary>Hint 1: full Jacobians in one call</summary><code>torch.autograd.functional.jacobian(func, inputs)</code> returns the Jacobian of <code>func</code> at <code>inputs</code>; for a <code>[seq, d] -> [seq, d]</code> function it has shape <code>[seq, d, seq, d]</code>.</details>

In [ ]:
predicted_zero_fraction = ...  # commit a number in [0, 1] before you code


def full_jacobian(
    model: TinyGPT,
    h_layer: Float[Tensor, "seq d"],
    layer: int,
) -> Float[Tensor, "seq d seq d"]:
    """jac[t_out, :, t_in, :] = d h_final[t_out] / d h_layer[t_in], where h_final = model.forward_from(layer, h_layer)."""
    # TODO: Implement
    pass


def average_jacobian(jac: Float[Tensor, "seq d seq d"]) -> Float[Tensor, "d d"]:
    """Mean of the [d, d] blocks over causally-valid pairs t_out >= t_in (only those -- don't average in zeros)."""
    # TODO: Implement
    pass

In [ ]:
# --- Part 1 Validation ---
jac_p0 = full_jacobian(model, h_at(prompts[0]), LAYER)
assert jac_p0.shape == (SEQ, D_MODEL, SEQ, D_MODEL), f"Shape: expected {(SEQ, D_MODEL, SEQ, D_MODEL)}, got {tuple(jac_p0.shape)}"

# Prediction reveal (before any raising assert -- the gap IS the signal)
block_norms = jac_p0.permute(0, 2, 1, 3).flatten(2).norm(dim=-1)  # [t_out, t_in]
actual_zero_fraction = (block_norms == 0).double().mean().item()
print(f"  You predicted zero-block fraction {predicted_zero_fraction}; actual {actual_zero_fraction:.3f}")

# Causality invariant
t_out = torch.arange(SEQ).view(-1, 1)
t_in = torch.arange(SEQ).view(1, -1)
n_future = int((t_out < t_in).sum())
assert (block_norms[t_out < t_in] == 0).all(), "causal attention means h_final[t_out] cannot depend on h_layer[t_in] when t_in > t_out"
print(f"  Causality: all {n_future} future->past blocks are exactly zero -- correct")

# Reference match: row-by-row torch.autograd.grad
h = h_at(prompts[0]).clone().requires_grad_(True)
h_final = model.forward_from(LAYER, h)
ref = torch.zeros(SEQ, D_MODEL, SEQ, D_MODEL)
for t in range(SEQ):
    for i in range(D_MODEL):
        ref[t, i] = torch.autograd.grad(h_final[t, i], h, retain_graph=True)[0]
max_diff = (jac_p0 - ref).abs().max()
try:
    assert torch.allclose(jac_p0, ref, atol=1e-9), f"Max diff: {max_diff:.2e} (threshold: 1e-9)"
except AssertionError:
    print(f"  YOUR block (0,0) corner: {jac_p0[0, :2, 0, :2].flatten().tolist()}")
    print(f"  EXPECTED:                {ref[0, :2, 0, :2].flatten().tolist()}")
    raise
print(f"  Reference match -- correct (max diff: {max_diff:.2e})")

# Independent reference for the causal-pair average
ref_avg = torch.stack([jac_p0[a, :, b, :] for a in range(SEQ) for b in range(SEQ) if a >= b]).mean(dim=0)
assert torch.allclose(average_jacobian(jac_p0), ref_avg, atol=1e-12), "the average must run over exactly the 21 causally-valid blocks"
print("  Causal-pair average -- correct")

# The corpus-averaged lens matrix, reused by every later part
J_avg = torch.stack([average_jacobian(full_jacobian(model, h_at(p), LAYER)) for p in prompts]).mean(dim=0)
print(f"  J_avg: shape {tuple(J_avg.shape)}, norm {J_avg.norm():.3f}, mean {J_avg.mean():.5f}, std {J_avg.std():.5f}")

print("\nPart 1 complete.")

## Part 2: Reading the Lens (Recall: logit lens)
One read function covers both lenses: with $M = I$ it is the logit lens you built in llm/10-Create-Embeddings-out-of-an-LLM; with $M = J_\ell$ it is the Jacobian lens. Reproduce it from memory -- your old worked solution is in that problem's `_solutions/` folder; opening it ends the rep, so only consult it after you've produced or genuinely failed an attempt.

**Predict before you code**: Commit the output shape below. And in prose: at a *middle* layer of a trained model, which of the two reads do you expect to rank plausible next tokens higher, and why does the final-layer basis matter?

In [ ]:
predicted_probs_shape = (...)  # commit the shape before you code


def lens_probs(
    h: Float[Tensor, "d"],
    M: Float[Tensor, "d d"],
    W_U: Float[Tensor, "vocab d"],
) -> Float[Tensor, "vocab"]:
    # Implement from memory: softmax over W_U @ layer_norm(M @ h)
    pass

In [ ]:
# --- Part 2 Validation ---
h_probe = h_at(prompts[0])[-1]  # layer-2 residual at the last position
p_jlens = lens_probs(h_probe, J_avg, W_U_w)
p_logit = lens_probs(h_probe, torch.eye(D_MODEL), W_U_w)

print(f"  You predicted probs shape {predicted_probs_shape}; actual {tuple(p_jlens.shape)}")
assert p_jlens.shape == (VOCAB,), f"Shape: expected {(VOCAB,)}, got {tuple(p_jlens.shape)}"
assert (p_jlens >= 0).all() and abs(p_jlens.sum().item() - 1.0) < 1e-9, "lens output must be a probability distribution (softmax over vocab)"
print(f"  Distribution: sum {p_jlens.sum():.6f}, min {p_jlens.min():.2e} -- correct")

ref = F.softmax(W_U_w @ F.layer_norm(J_avg @ h_probe, (D_MODEL,)), dim=-1)
max_diff = (p_jlens - ref).abs().max()
try:
    assert torch.allclose(p_jlens, ref, atol=1e-9), f"Max diff: {max_diff:.2e} (threshold: 1e-9)"
except AssertionError:
    print(f"  YOUR top-5:     {p_jlens.topk(5).indices.tolist()}")
    print(f"  EXPECTED top-5: {ref.topk(5).indices.tolist()}")
    print("  (check the order of operations: M @ h first, THEN layer_norm, THEN W_U)")
    raise
print(f"  Reference match -- correct (max diff: {max_diff:.2e})")

jl_top = p_jlens.topk(5).indices.tolist()
ll_top = p_logit.topk(5).indices.tolist()
print(f"  J-lens top-5 tokens: {jl_top}")
print(f"  logit-lens top-5:    {ll_top}")
print(f"  overlap: {len(set(jl_top) & set(ll_top))}/5 (random model, no semantics -- but note the two reads already disagree)")

print("\nPart 2 complete.")

## Part 3: J-lens Vectors and Steering
The lens's row for token $t$ is a *direction in layer-$\ell$ space*: $v_t = J_\ell^\top u_t$. Adding $\alpha v_t$ to the activation is the paper's steering intervention.

**Insight**: $v_t = J^\top u_t$ is exactly the gradient of the downstream readout $u_t^\top h_{\mathrm{final},t'}$ with respect to $h_{\ell,t}$ — the direction you *read* a concept from is the same direction you *write* it with. In one line, explain why the chain rule makes these the same vector.

**Predict before you run**: for $\alpha > 0$, will the downstream readout for token $t$ go up or down? Commit the sign below.

In [ ]:
predicted_dreadout_sign = ...  # commit +1 or -1 before you code


def jlens_vector(J: Float[Tensor, "d d"], u_t: Float[Tensor, "d"]) -> Float[Tensor, "d"]:
    # Implement from memory
    pass


def steer(h: Float[Tensor, "d"], v: Float[Tensor, "d"], alpha: float) -> Float[Tensor, "d"]:
    # Implement from memory
    pass

In [ ]:
# --- Part 3 Validation ---
T_IN, T_OUT, TOK = 2, 5, 7
J_block = jac_p0[T_OUT, :, T_IN, :]  # exact Jacobian block for prompt 0 -- steering theory is exact here
u_t = W_U_w[TOK]
v = jlens_vector(J_block, u_t)
assert v.shape == (D_MODEL,), f"Shape: expected {(D_MODEL,)}, got {tuple(v.shape)}"

# Chain-rule reference: v_t must equal the autograd gradient of the downstream readout
h = h_at(prompts[0]).clone().requires_grad_(True)
readout = u_t @ model.forward_from(LAYER, h)[T_OUT]
g = torch.autograd.grad(readout, h)[0][T_IN]
max_diff = (v - g).abs().max()
try:
    assert torch.allclose(v, g, atol=1e-9), f"Max diff: {max_diff:.2e} (threshold: 1e-9)"
except AssertionError:
    print(f"  YOUR v[:4]:     {v[:4].tolist()}")
    print(f"  EXPECTED g[:4]: {g[:4].tolist()}")
    print("  (if these look unrelated, check which side of J you put u_t on -- rows of W_U J are J^T u_t)")
    raise
print("  v_t == autograd gradient of the downstream readout -- chain rule verified")

# First-order steering check
alpha = 1e-3
base = readout.item()
h0 = h_at(prompts[0])
h_steered = h0.clone()
h_steered[T_IN] = steer(h0[T_IN], v, alpha)
d_actual = (u_t @ model.forward_from(LAYER, h_steered)[T_OUT]).item() - base
d_first_order = alpha * v.norm().pow(2).item()
sign = 1 if d_actual > 0 else -1
print(f"  You predicted sign {predicted_dreadout_sign}; actual {sign:+d}")
print(f"  d_readout actual {d_actual:.3e} vs first-order alpha*||v||^2 = {d_first_order:.3e}")
rel_err = abs(d_actual - d_first_order) / abs(d_first_order)
assert rel_err < 0.05, f"first-order prediction should match within 5% at alpha=1e-3 (rel err {rel_err:.4f}) -- did steer() add alpha*v?"
print(f"  First-order match -- correct (rel err {rel_err:.2e})")

print("\nPart 3 complete.")

## Part 4: Lens-Coordinate Patching (concept swap)
The paper's cleanest intervention: read an activation's coordinates in a small concept basis, edit the coordinates, write the edit back. Swapping two concepts' coordinates redirects what the model reports.

**Insight**: because $V^+ V = I_2$ whenever $v_s, v_t$ are linearly independent, adding $V(\sigma(c) - c)$ moves the lens coordinates from $c$ to $\sigma(c)$ while leaving every direction that $V^+$ ignores untouched. In one line: why must patching twice return the original $h$?

**Predict before you code** (prose): the patch changes $h$ by a vector inside span($v_s, v_t$). Does $\|h\|$ stay the same? Which readouts other than tokens $s, t$ can change?

<details><summary>Hint 1: pseudoinverse</summary><code>torch.linalg.pinv(V)</code> gives the Moore-Penrose pseudoinverse; for a full-column-rank V, <code>pinv(V) @ V = I</code>.</details>

In [ ]:
def lens_coordinate_patch(
    h: Float[Tensor, "d"],
    v_s: Float[Tensor, "d"],
    v_t: Float[Tensor, "d"],
) -> Float[Tensor, "d"]:
    """Swap the (v_s, v_t) lens coordinates of h.

    Step 1: stack the two vectors as the COLUMNS of V, shape [d, 2].
    Step 2: read coordinates c = V+ @ h  (pseudoinverse).
    # Hint: torch.linalg.pinv
    Step 3: write back: h + V @ (swapped(c) - c).
    """
    pass

In [ ]:
# --- Part 4 Validation ---
S_TOK, T_TOK = 3, 7
v_s = jlens_vector(J_avg, W_U_w[S_TOK])
v_t = jlens_vector(J_avg, W_U_w[T_TOK])
V = torch.stack([v_s, v_t], dim=1)
c_before = torch.linalg.pinv(V) @ h_probe

h_patched = lens_coordinate_patch(h_probe, v_s, v_t)
assert h_patched.shape == (D_MODEL,), f"Shape: expected {(D_MODEL,)}, got {tuple(h_patched.shape)}"

c_after = torch.linalg.pinv(V) @ h_patched
print(f"  coordinates before: {[round(x, 4) for x in c_before.tolist()]}")
print(f"  coordinates after:  {[round(x, 4) for x in c_after.tolist()]}")
max_diff = (c_after - c_before.flip(0)).abs().max()
try:
    assert torch.allclose(c_after, c_before.flip(0), atol=1e-9), f"Max diff: {max_diff:.2e} (threshold: 1e-9)"
except AssertionError:
    print("  EXPECTED the two coordinates exactly exchanged -- check the sign and order of (swapped(c) - c)")
    raise
print(f"  Lens coordinates exactly swapped -- correct (max diff: {max_diff:.2e})")

# Involution invariant: patching twice must be the identity
h_twice = lens_coordinate_patch(h_patched, v_s, v_t)
assert torch.allclose(h_twice, h_probe, atol=1e-9), "swap is an involution: applying it twice must return the original activation"
print("  Double patch == identity -- correct")

# Diagnostics: what moved
print(f"  ||h_patched - h||: {(h_patched - h_probe).norm():.4f} (all of it inside span(v_s, v_t))")
print(f"  readout v_s . h: {v_s @ h_probe:.4f} -> {v_s @ h_patched:.4f}")
print(f"  readout v_t . h: {v_t @ h_probe:.4f} -> {v_t @ h_patched:.4f}")

print("\nPart 4 complete.")

## Part 5: J-space Decomposition (non-negative matching pursuit)
How much of an activation is "verbalizable"? The paper defines the J-space as sparse non-negative combinations of J-lens vectors and finds only ~25 concepts active at a time. Measure that with greedy pursuit.

**Insight**: greedy pursuit explains the residual one concept at a time — pick the atom with the largest positive correlation, subtract its projection, repeat; non-negativity encodes that a concept can be present, never anti-present. In one line: why can the residual norm never increase?

The algorithm (the idea — the code is yours):
1. residual `r = h`; record `||r||`.
2. Up to `k` times: score every atom by `r . v_i / ||v_i||` (scale-invariant correlation); stop early if the best score is <= 0.
3. Accept the best atom `i*`: add its least-squares coefficient `(r . v_i*) / ||v_i*||^2` to `coeffs[i*]`, subtract the projection from `r`, record `||r||`.

**Predict before you code**: with 100 atoms living in a 32-dimensional space, what fraction of $\|h\|^2$ do you expect the $k=25$ decomposition to explain? Commit a number in [0, 1].

In [ ]:
predicted_fraction_k25 = ...  # commit a number in [0, 1] before you code


def jspace_decompose(
    h: Float[Tensor, "d"],
    atoms: Float[Tensor, "n d"],
    k: int,
) -> tuple[Float[Tensor, "n"], list[float]]:
    """Greedy non-negative matching pursuit of h onto `atoms`. Returns (coeffs, residual_norm_history)."""
    # TODO: Implement (algorithm in the header above)
    pass

In [ ]:
# --- Part 5 Validation ---
atoms = W_U_w @ J_avg  # atoms[i] = J_avg^T u_i -- one J-lens vector per vocab token, [VOCAB, D_MODEL]
coeffs, hist = jspace_decompose(h_probe, atoms, k=25)

# Prediction reveal (before any raising assert)
fraction = 1 - hist[-1] ** 2 / hist[0] ** 2
print(f"  You predicted J-space fraction {predicted_fraction_k25}; actual {fraction:.3f}")
print("  (a large gap is the most valuable moment of the session)")

assert coeffs.shape == (VOCAB,), f"Shape: expected {(VOCAB,)}, got {tuple(coeffs.shape)}"
assert (coeffs >= 0).all(), "non-negative pursuit: a concept can be present, never anti-present"
n_active = int((coeffs > 0).sum())
assert n_active <= 25, f"at most k=25 atoms may be active, got {n_active}"
print(f"  Sparsity: {n_active} active atoms (k=25), all coefficients >= 0 -- correct")

drops = [hist[i + 1] - hist[i] for i in range(len(hist) - 1)]
assert all(d <= 1e-9 for d in drops), "each greedy projection can only shrink the residual (subtract, never add, energy)"
print(f"  Residual norm monotone: {hist[0]:.3f} -> {hist[-1]:.3f} over {len(hist) - 1} greedy steps -- correct")

recon = coeffs @ atoms
resid_norm = (h_probe - recon).norm().item()
assert abs(resid_norm - hist[-1]) < 1e-6, "coeffs must reproduce the pursuit: h = coeffs @ atoms + final residual"
assert 0.0 <= fraction <= 1.0, "explained fraction of ||h||^2 must be in [0, 1]"
print(f"  Reconstruction consistent -- correct (final residual {resid_norm:.4f})")

for kk in (1, 5, 10, 25):
    _, hk = jspace_decompose(h_probe, atoms, k=kk)
    print(f"    k={kk:>2}: J-space fraction {1 - hk[-1] ** 2 / hk[0] ** 2:.3f}")

print("\nPart 5 complete.")

## Session Debrief

Write your answers into the code cell below (typing is overt retrieval — far stronger than answering "in your head"). Don't scroll up.
1. What is the formula for the averaged Jacobian $J_\ell$, including which position pairs enter the average?
2. Which single PyTorch call builds a full Jacobian matrix of a function, and what shape does it return for a `[seq, d] -> [seq, d]` function?
3. What is the J-lens vector $v_t$ in terms of $J$ and $u_t$ — and what *other* quantity is it exactly equal to?

**Check yourself**: your worked solution is in `_solutions/` — open it (and the paper) to grade your answers. Opening it ends the retrieval rep, so answer first.

**Challenge**: Close this notebook, open a blank one, and rewrite Part 5 (matching pursuit) from scratch without looking back.

In [ ]:
debrief = """
1.
2.
3.
"""  # type your recall here before checking _solutions/

In [ ]:
# --- Session log: fill the `___` then run (appends one line to .practice-log.jsonl) ---
import json, datetime, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / ".git").exists() or (p / ".spaced-repetition.json").exists())
record = {
    "problem": "papers/jacobian-lens",
    "date": datetime.date.today().isoformat(),
    "budget_min": 60,
    "actual_min": ___,                                  # how long it really took
    "parts": [                                          # n + tier pre-filled; set outcome
        {"n": 1, "tier": 1, "outcome": "___"},          # unaided | hint | solution | failed
        {"n": 2, "tier": 3, "outcome": "___"},
        {"n": 3, "tier": 3, "outcome": "___"},
        {"n": 4, "tier": 1, "outcome": "___"},
        {"n": 5, "tier": 2, "outcome": "___"},
    ],
    "difficulty": ___,                                  # 1 (trivial) .. 5 (over my head)
    "stuck": "___",                                     # one phrase: where you got stuck
}
with open(_root / ".practice-log.jsonl", "a") as f:
    f.write(json.dumps(record) + "\n")
print("logged ->", _root / ".practice-log.jsonl")

## Extension (Optional)
Try one of these variations:
- **Workspace profile**: compute the k=25 J-space fraction at every depth (layers 0-4). The paper finds an inverted-U ("workspace") shape in trained models — what does a random-init model give, and why?
- **Ablate the J-space**: project `h_probe` onto the orthogonal complement of the 10 atoms Part 5 selected first, and re-run the lens read. How much does the top-1 token change vs. removing 10 *random* directions?
- **Break it on purpose**: in Part 1, average over ALL 36 blocks instead of the 21 valid ones. Which later validation fails first, and why exactly that one?

## Anki Cards

Add these to your deck:

**Card 1**
Front: Logit-lens reads look like garbage at middle layers; the workspace paper's fix reads through what matrix?
Back: J_l = E[dh_final/dh_l]

**Card 2**
Front: You want the layer-l direction that maximally boosts token t's downstream readout (first order). Which vector?
Back: v_t = J_l^T u_t

**Card 3**
Front: Your concept-swap patch drifts h after applying it twice — which algebraic property did your write-back break?
Back: involution (V+V = I)